In [7]:
import pandas as pd
import numpy as np
import nltk
import re
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
import spacy

nltk.download('punkt')
nltk.download('stopwords')

df = pd.read_csv("../data/train.csv")
df = df.dropna(subset=['text', 'class'])

stop_words = set(stopwords.words("catalan"))
nlp = spacy.load("ca_core_news_sm")

def spacy_tokenize(text):
    doc = nlp(text.lower())
    return [token.text for token in doc if not token.is_punct and not token.is_space and token.text not in stop_words]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r"[^\w\s]", '', text)
    tokens = spacy_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    return tokens

df['tokens'] = df['text'].apply(preprocess)

w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=3, workers=4, epochs=20)

def get_doc_vector(tokens):
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    if vecs:
        return np.mean(vecs, axis=0)
    else:
        return np.zeros(w2v_model.vector_size)

df['doc_vec'] = df['tokens'].apply(get_doc_vector)

X = np.vstack(df['doc_vec'].values)
le = LabelEncoder()
y = le.fit_transform(df['class'])
num_classes = len(np.unique(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        return self.fc2(out)

model = TextClassifier(input_dim=100, hidden_dim=64, output_dim=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        _, predicted = torch.max(preds, 1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print(f"\nTest Accuracy: {correct / total:.4f}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arhip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arhip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Epoch 1: Loss = 1778.7278
Epoch 2: Loss = 1253.7651
Epoch 3: Loss = 1178.8219
Epoch 4: Loss = 1137.9045
Epoch 5: Loss = 1110.3367
Epoch 6: Loss = 1093.1462
Epoch 7: Loss = 1081.6743
Epoch 8: Loss = 1072.0239
Epoch 9: Loss = 1058.6029
Epoch 10: Loss = 1057.4517

Test Accuracy: 0.9425
